## Importación de librerías

En esta sección se importan las librerías necesarias para el proceso de **ETL (Extracción, Transformación y Carga)** y la manipulación de datos.

- **pandas**: Se utiliza para la manipulación y análisis de datos en estructuras tipo DataFrame, permitiendo limpiar, transformar y preparar la información antes de cargarla al Data Warehouse.
- **sqlalchemy**: Permite crear una conexión entre Python y la base de datos **PostgreSQL**, facilitando la carga de los datos procesados hacia las tablas del modelo estrella.

Estas librerías son fundamentales para automatizar el flujo de datos desde las fuentes originales hasta el sistema de almacenamiento final.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

## Configuración de la conexión a la base de datos

En esta sección se definen los parámetros necesarios para establecer la conexión con la base de datos **PostgreSQL** donde se encuentra implementado el **modelo estrella del Data Warehouse**.

Se especifican las siguientes variables:

- **DB_USER**: Usuario de la base de datos.
- **DB_PASSWORD**: Contraseña del usuario.
- **DB_HOST**: Dirección del servidor de la base de datos (en este caso `localhost` porque se ejecuta de forma local).
- **DB_PORT**: Puerto utilizado por PostgreSQL (por defecto `5432`).
- **DB_NAME**: Nombre de la base de datos que contiene el modelo estrella (`modelo_estrella_ventas`).

Posteriormente se utiliza la función `create_engine()` de **SQLAlchemy** para construir el motor de conexión que permitirá a Python interactuar con PostgreSQL. Este motor será utilizado más adelante para **cargar las dimensiones y la tabla de hechos** generadas durante el proceso ETL.

Finalmente, se imprime un mensaje de confirmación para verificar que la conexión fue establecida correctamente.

In [ ]:
DB_USER = "postgres"
DB_PASSWORD = "1287985dbc"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "modelo_estrella_ventas"

# Crear conexión usando SQLAlchemy
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Conexión a PostgreSQL exitosa")

Conexión a PostgreSQL exitosa


## Carga del dataset

En este paso se realiza la **carga del conjunto de datos original** que contiene la información de las transacciones de compras realizadas por los clientes.

Para ello se utiliza la función `read_csv()` de la librería **Pandas**, que permite leer archivos en formato `.csv` y almacenarlos en una estructura de tipo **DataFrame**, la cual facilita la manipulación y análisis de los datos.

El archivo cargado se denomina **`customer_shopping_data.csv`** y contiene información relevante como:

- Identificador de la factura (`invoice_no`)
- Identificador del cliente (`customer_id`)
- Género
- Edad
- Categoría del producto
- Cantidad comprada
- Precio
- Método de pago
- Fecha de la compra
- Centro comercial (`shopping_mall`)

Posteriormente se utiliza `df.head(5)` para visualizar **las primeras cinco filas del dataset**, lo cual permite verificar que los datos se cargaron correctamente y observar su estructura inicial.

In [4]:
df = pd.read_csv("customer_shopping_data.csv")
df.head(5)

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon


## Análisis estadístico inicial del dataset

En este paso se utiliza la función `describe()` de **Pandas** para obtener un **resumen estadístico de las variables numéricas** presentes en el conjunto de datos.

Este análisis permite observar métricas importantes como:

- **count**: número de registros disponibles para cada variable.
- **mean**: valor promedio.
- **std**: desviación estándar, que indica qué tan dispersos están los datos respecto al promedio.
- **min**: valor mínimo registrado.
- **25%**, **50%**, **75%**: cuartiles que ayudan a entender la distribución de los datos.
- **max**: valor máximo registrado.

Las columnas numéricas analizadas incluyen principalmente:

- **Edad del cliente (`age`)**
- **Cantidad de productos comprados (`quantity`)**
- **Precio del producto (`price`)**

Este paso es importante porque permite **detectar valores atípicos, rangos de datos y posibles inconsistencias**, lo cual ayuda en la etapa de **limpieza y transformación de datos dentro del proceso ETL** antes de construir el **modelo de datos tipo estrella**.

In [5]:
df.describe()

,age,quantity,price
count,99457.000000,99457.000000,99457.000000
mean,43.427089,3.003429,689.256321
std,14.990054,1.413025,941.184567
min,18.000000,1.000000,5.230000
25%,30.000000,2.000000,45.450000
50%,43.000000,3.000000,203.300000
75%,56.000000,4.000000,1200.320000
max,69.000000,5.000000,5250.000000


## Estructura del dataset

En esta etapa se utiliza la función `df.info()` de **Pandas** para analizar la **estructura general del dataset**. Esta función permite conocer información importante sobre el conjunto de datos antes de iniciar el proceso de transformación para el modelo estrella.

La salida muestra:

- **Número total de registros:** 99,457 filas.
- **Número total de columnas:** 10 variables.
- **Tipos de datos de cada columna.**
- **Cantidad de valores no nulos por columna.**
- **Uso de memoria del DataFrame.**

### Columnas identificadas

El dataset contiene las siguientes variables:

- **invoice_no**: identificador de la factura o transacción.
- **customer_id**: identificador del cliente.
- **gender**: género del cliente.
- **age**: edad del cliente.
- **category**: categoría del producto comprado.
- **quantity**: cantidad de productos comprados en la transacción.
- **price**: precio del producto.
- **payment_method**: método de pago utilizado.
- **invoice_date**: fecha en que se realizó la compra.
- **shopping_mall**: centro comercial donde se realizó la compra.

### Observaciones importantes

A partir de esta inspección inicial se identifican algunos aspectos relevantes para el proceso ETL:

- La columna **`invoice_date`** se encuentra como tipo `object`, por lo que deberá convertirse a **tipo fecha (`datetime`)** para poder construir correctamente la **dimensión de tiempo** en el modelo estrella.
- Varias columnas categóricas (`gender`, `category`, `payment_method`, `shopping_mall`) posteriormente se transformarán en **dimensiones del modelo de datos**.
- Las variables **`quantity`** y **`price`** serán utilizadas para calcular la métrica **`total_amount`**, que será almacenada en la **tabla de hechos de ventas**.

Este análisis permite comprender la estructura del dataset y definir los pasos necesarios para su **limpieza, transformación y carga en el modelo dimensional**.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      99457 non-null  object 
 1   customer_id     99457 non-null  object 
 2   gender          99457 non-null  object 
 3   age             99457 non-null  int64  
 4   category        99457 non-null  object 
 5   quantity        99457 non-null  int64  
 6   price           99457 non-null  float64
 7   payment_method  99457 non-null  object 
 8   invoice_date    99457 non-null  object 
 9   shopping_mall   99457 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 7.6+ MB


## Conversión de variables categóricas

En este paso se realiza la **conversión de algunas columnas a tipo `category`** utilizando Pandas. Este tipo de dato es especialmente útil para variables que representan **categorías o valores repetitivos**, ya que optimiza el uso de memoria y facilita posteriores operaciones de análisis.

Las columnas convertidas a tipo categórico son:

- **gender**
- **payment_method**
- **category**
- **shopping_mall**

Estas variables representan **atributos descriptivos** y no valores numéricos continuos, por lo que es más apropiado manejarlas como categorías.

### Beneficios de usar el tipo `category`

1. **Optimización de memoria:**  
   Pandas almacena internamente las categorías como códigos numéricos, reduciendo el consumo de memoria en datasets grandes.

2. **Mejor rendimiento en análisis:**  
   Operaciones como `groupby`, filtrado o agregaciones pueden ejecutarse de forma más eficiente.

3. **Claridad semántica:**  
   Permite identificar fácilmente qué columnas representan **atributos categóricos**, lo cual es útil durante el diseño del **modelo estrella**, ya que estas columnas suelen convertirse en **dimensiones**.

### Relación con el modelo estrella

Estas columnas categóricas posteriormente serán utilizadas para construir **dimensiones del modelo dimensional**, por ejemplo:

- **gender** → atributo dentro de la **dimensión cliente**
- **payment_method** → posible **dimensión método de pago**
- **category** → **dimensión categoría de producto**
- **shopping_mall** → **dimensión centro comercial**

Esta transformación es parte del proceso de **preparación de los datos antes del ETL** hacia el data warehouse.

In [7]:
df["gender"] = df["gender"].astype("category")
df["payment_method"] = df["payment_method"].astype("category") 
df["category"] = df["category"].astype("category")
df["shopping_mall"] = df["shopping_mall"].astype("category")



## Conversión de la columna de fecha

En este paso se transforma la columna **`invoice_date`** al tipo de dato **`datetime`** utilizando la función `pd.to_datetime()` de **Pandas**.

El dataset original almacena las fechas como **cadenas de texto (`object`)**, lo cual impide realizar operaciones de análisis temporal. Por esta razón es necesario convertirlas a un formato de fecha adecuado.

```python
df["invoice_date"] = pd.to_datetime(df["invoice_date"], dayfirst=True)
````

El parámetro **`dayfirst=True`** se utiliza porque el formato de las fechas en el dataset sigue el patrón:

```
DD/MM/YYYY
```

Por ejemplo:

* `16/05/2021`
* `24/10/2021`

Sin este parámetro, Pandas podría interpretar incorrectamente el orden de **día y mes**, generando errores durante la conversión.

### Importancia de esta transformación

Convertir la fecha permite posteriormente:

* Extraer **día, mes y año**
* Identificar **fines de semana**
* Realizar **análisis temporales de ventas**
* Construir correctamente la **dimensión de fecha (`dim_date`)** en el modelo estrella

Esta transformación es fundamental dentro del proceso **ETL**, ya que la dimensión de tiempo es una de las dimensiones más importantes en los **data warehouses orientados a análisis de ventas**.

```

In [8]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"], dayfirst=True)

## Extracción de componentes de la fecha

Una vez que la columna **`invoice_date`** fue convertida al tipo de dato `datetime`, ahora es posible **extraer componentes específicos de la fecha** como el día, el mes y el año.

Esto se realiza utilizando el accesor **`.dt`** de Pandas, que permite trabajar con propiedades de fechas dentro de una columna.

```python
df["day"] = df["invoice_date"].dt.day
df["month"] = df["invoice_date"].dt.month
df["year"] = df["invoice_date"].dt.year
````

### ¿Qué hace cada línea?

* **`dt.day`** → extrae el **día del mes** de la fecha.
* **`dt.month`** → extrae el **mes**.
* **`dt.year`** → extrae el **año**.

Por ejemplo, si una fila tiene la fecha:

```
16/05/2021
```

Después de la transformación se obtendrá:

| invoice_date | day | month | year |
| ------------ | --- | ----- | ---- |
| 2021-05-16   | 16  | 5     | 2021 |

### ¿Por qué hacemos esto?

En un **Data Warehouse**, es común descomponer las fechas en varias columnas porque esto facilita mucho las **consultas analíticas**. Por ejemplo:

* Analizar **ventas por mes**
* Analizar **ventas por año**
* Analizar **ventas por día específico**

Además, estas columnas serán utilizadas posteriormente para construir la **dimensión de tiempo (`dim_date`)** dentro del **modelo estrella**, permitiendo realizar análisis temporales de forma eficiente.

```

In [9]:
df["day"] = df["invoice_date"].dt.day
df["month"] = df["invoice_date"].dt.month
df["year"] = df["invoice_date"].dt.year

## Creación del indicador de fin de semana

En este paso se crea una nueva columna llamada **`is_weekend`**, que indica si una venta ocurrió durante el **fin de semana**.

```python
df["is_weekend"] = df["invoice_date"].dt.weekday >= 5
````

### ¿Cómo funciona?

Se utiliza nuevamente el accesor **`.dt`** de Pandas para acceder a propiedades de la fecha.

* **`dt.weekday`** devuelve un número del **0 al 6** que representa el día de la semana:

| Número | Día       |
| ------ | --------- |
| 0      | Lunes     |
| 1      | Martes    |
| 2      | Miércoles |
| 3      | Jueves    |
| 4      | Viernes   |
| 5      | Sábado    |
| 6      | Domingo   |

Luego se aplica la condición:

```python
>= 5
```

Esto significa:

* **5 (sábado)** → `True`
* **6 (domingo)** → `True`
* **0 a 4 (lunes a viernes)** → `False`

Por lo tanto, la nueva columna **`is_weekend`** será un **valor booleano**:

| invoice_date | weekday | is_weekend |
| ------------ | ------- | ---------- |
| 2021-05-16   | 6       | True       |
| 2021-05-17   | 0       | False      |

### ¿Por qué es útil esta columna?

Este campo es muy útil para **análisis de comportamiento de ventas**, por ejemplo:

* Comparar **ventas entre semana vs fin de semana**
* Analizar **afluencia en centros comerciales**
* Identificar **patrones de compra de los clientes**

Además, esta variable será parte de la **dimensión de fecha (`dim_date`)** dentro del **modelo estrella**, permitiendo realizar filtros analíticos de forma eficiente en las consultas SQL.

```

In [10]:
df["is_weekend"] = df["invoice_date"].dt.weekday >= 5


## Creación de la métrica `total_amount`

En este paso se crea una nueva columna llamada **`total_amount`**, que representa el **valor total de cada venta**.

```python
df["total_amount"] = df["quantity"] * df["price"]
````

### ¿Qué hace esta operación?

Se multiplica:

* **`quantity`** → cantidad de productos comprados
* **`price`** → precio unitario del producto

El resultado es el **valor total de la transacción**.

### Ejemplo

| quantity | price   | total_amount |
| -------- | ------- | ------------ |
| 5        | 1500.40 | 7502.00      |
| 3        | 1800.51 | 5401.53      |
| 1        | 300.08  | 300.08       |

Es decir:

```text
total_amount = quantity × price
```

### ¿Por qué es importante esta columna?

En los **data warehouses**, la tabla de hechos contiene **métricas de negocio** que se pueden analizar y agregar.

En este proyecto:

* **`quantity`** → métrica (cantidad vendida)
* **`total_amount`** → métrica principal (valor de ventas)

Estas métricas permitirán realizar consultas analíticas como:

* **Ventas totales por categoría**
* **Ventas por centro comercial**
* **Ventas por mes**
* **Ventas en fin de semana**

Por lo tanto, **`total_amount` será una de las medidas principales dentro de la tabla de hechos `fact_sales`** del modelo estrella.

```

In [11]:
df["total_amount"] = df["quantity"] * df["price"]

## Exploración de valores únicos en variables categóricas

En este paso se analizan los **valores únicos** presentes en algunas columnas categóricas del dataset utilizando la función **`unique()`** de Pandas.

```python
df["gender"].unique()
df["payment_method"].unique()
````

### ¿Qué hace `unique()`?

La función **`unique()`** devuelve todos los **valores distintos** que aparecen en una columna. Esto es muy útil durante el **análisis exploratorio de datos (EDA)** porque permite:

* Identificar **las categorías existentes**
* Detectar **errores o inconsistencias**
* Conocer cuántos **valores diferentes** tiene una variable

### Resultados observados

Para la columna **`payment_method`** se encontraron tres valores únicos:

* `Cash`
* `Credit Card`
* `Debit Card`

Esto indica que existen **tres métodos de pago distintos** registrados en el dataset.

### ¿Por qué es importante este paso?

Analizar los valores únicos es clave antes de construir el **modelo estrella**, porque:

1. Permite identificar **posibles dimensiones** del modelo.
2. Ayuda a detectar **valores inconsistentes o duplicados** (por ejemplo: "credit card", "Credit card", "Credit Card").
3. Facilita la creación de **tablas de dimensión limpias y controladas**.

En este caso, la columna **`payment_method`** puede convertirse en una **dimensión de método de pago**, mientras que **`gender`** se utilizará como atributo dentro de la **dimensión cliente**.

```

In [12]:
df["gender"].unique()
df["payment_method"].unique()


['Credit Card', 'Debit Card', 'Cash']
Categories (3, object): ['Cash', 'Credit Card', 'Debit Card']

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   invoice_no      99457 non-null  object        
 1   customer_id     99457 non-null  object        
 2   gender          99457 non-null  category      
 3   age             99457 non-null  int64         
 4   category        99457 non-null  category      
 5   quantity        99457 non-null  int64         
 6   price           99457 non-null  float64       
 7   payment_method  99457 non-null  category      
 8   invoice_date    99457 non-null  datetime64[ns]
 9   shopping_mall   99457 non-null  category      
 10  day             99457 non-null  int32         
 11  month           99457 non-null  int32         
 12  year            99457 non-null  int32         
 13  is_weekend      99457 non-null  bool          
 14  total_amount    99457 non-null  float64       
dtypes:

In [14]:
df.columns

Index(['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity',
       'price', 'payment_method', 'invoice_date', 'shopping_mall', 'day',
       'month', 'year', 'is_weekend', 'total_amount'],
      dtype='object')

In [15]:
duplicate = df[df.duplicated(subset=["customer_id", "invoice_date", "total_amount"])]
duplicate

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall,day,month,year,is_weekend,total_amount


In [16]:
dim_customer = df[["customer_id", "gender", "age"]].drop_duplicates()
dim_category = df[["category"]].drop_duplicates()
dim_category.columns = ["category_name"]
dim_payment = df[["payment_method"]].drop_duplicates()
dim_mall = df[["shopping_mall"]].drop_duplicates()
dim_date = df[["invoice_date","day","month","year","is_weekend"]].drop_duplicates()
dim_date.columns = ["full_date","day","month","year","is_weekend"]


In [17]:
# --------------------------------
# 5. CARGAR DIMENSIONES (LOAD)
# --------------------------------

print("Cargando dimensiones...")

dim_customer.to_sql(
    "dim_customer",
    engine,
    if_exists="append",
    index=False
)

dim_category.to_sql(
    "dim_category",
    engine,
    if_exists="append",
    index=False
)

dim_payment.to_sql(
    "dim_payment",
    engine,
    if_exists="append",
    index=False
)

dim_mall.to_sql(
    "dim_mall",
    engine,
    if_exists="append",
    index=False
)

dim_date.to_sql(
    "dim_date",
    engine,
    if_exists="append",
    index=False
)

print("Dimensiones cargadas correctamente")

Cargando dimensiones...
Dimensiones cargadas correctamente


In [18]:
# --------------------------------
# 6. OBTENER CLAVES DE DIMENSIONES
# --------------------------------

# Leer las dimensiones desde PostgreSQL
dim_customer_db = pd.read_sql("SELECT * FROM dim_customer", engine)
dim_category_db = pd.read_sql("SELECT * FROM dim_category", engine)
dim_payment_db = pd.read_sql("SELECT * FROM dim_payment", engine)
dim_mall_db = pd.read_sql("SELECT * FROM dim_mall", engine)
dim_date_db = pd.read_sql("SELECT * FROM dim_date", engine)

print("Dimensiones recuperadas desde la base de datos")


Dimensiones recuperadas desde la base de datos


In [20]:
# Convertir a datetime
dim_date_db["full_date"] = pd.to_datetime(dim_date_db["full_date"])

In [21]:
# --------------------------------
# 7. CREAR LA TABLA DE HECHOS
# --------------------------------

# Merge con dimensión cliente
fact_sales = df.merge(
    dim_customer_db,
    on=["customer_id","gender","age"]
)

# Merge con categoría
fact_sales = fact_sales.merge(
    dim_category_db,
    left_on="category",
    right_on="category_name"
)

# Merge con payment
fact_sales = fact_sales.merge(
    dim_payment_db,
    on="payment_method"
)

# Merge con mall
fact_sales = fact_sales.merge(
    dim_mall_db,
    on="shopping_mall"
)

# Merge con fecha
fact_sales = fact_sales.merge(
    dim_date_db,
    left_on="invoice_date",
    right_on="full_date"
)


In [22]:
# --------------------------------
# 8. SELECCIONAR CAMPOS FACT
# --------------------------------

fact_sales = fact_sales[[
    "invoice_no",
    "customer_key",
    "date_key",
    "category_key",
    "payment_key",
    "mall_key",
    "quantity",
    "price",
    "total_amount"
]]

print("Tabla de hechos creada")

Tabla de hechos creada


In [23]:
# --------------------------------
# 9. CARGAR TABLA DE HECHOS
# --------------------------------

fact_sales.to_sql(
    "fact_sales",
    engine,
    if_exists="append",
    index=False
)

print("ETL COMPLETADO")
print("Datos cargados correctamente en PostgreSQL")

ETL COMPLETADO
Datos cargados correctamente en PostgreSQL
